# Mann-Whitney U Evaluation

## Setup

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

BASE_DIR = Path(".")

RESULT_FILES = {
    "CodeLlama": BASE_DIR / "CodeLlama_Automatic_Evaluation_Results.xlsx",
    "DeepSeek": BASE_DIR / "DeepSeek_Automatic_Evaluation_Results.xlsx",
    "Qwen": BASE_DIR / "Qwen_Automatic_Evaluation_Results.xlsx",
}

NUMERIC_COLUMNS = [
    "BLEU Zero",
    "ROUGE Zero",
    "METEOR Zero",
    "BLEU Few",
    "ROUGE Few",
    "METEOR Few",
    "BLEU Adv",
    "ROUGE Adv",
    "METEOR Adv",
]

STRATEGIES = {
    "Zero-shot": ["BLEU Zero", "ROUGE Zero", "METEOR Zero"],
    "Few-shot": ["BLEU Few", "ROUGE Few", "METEOR Few"],
    "Chain of Thought": ["BLEU Adv", "ROUGE Adv", "METEOR Adv"],
}


def load_balanced_results(sample_size, random_state):
    frames = []

    for model_name, file_path in RESULT_FILES.items():
        frame = pd.read_excel(file_path)
        frame = frame.sample(n=sample_size, random_state=random_state)
        frame["Model"] = model_name
        frames.append(frame)

    data = pd.concat(frames, ignore_index=True)

    for column in NUMERIC_COLUMNS:
        data[column] = (
            data[column]
            .astype(str)
            .str.replace(",", ".", regex=False)
        )
        data[column] = pd.to_numeric(data[column], errors="coerce")

    return data


## Mann-Whitney U Test

In [ ]:

df = load_balanced_results(sample_size=2357, random_state=335)

comparisons = [
    ("Zero-shot", "Few-shot"),
    ("Zero-shot", "Chain of Thought"),
    ("Few-shot", "Chain of Thought"),
]

results = []

for model in ["CodeLlama", "DeepSeek", "Qwen"]:
    model_df = df[df["Model"] == model]

    for metric_name, metric_index in zip(
        ["BLEU", "ROUGE", "METEOR"],
        [0, 1, 2],
    ):
        for strategy_1, strategy_2 in comparisons:
            column_1 = STRATEGIES[strategy_1][metric_index]
            column_2 = STRATEGIES[strategy_2][metric_index]

            data_1 = model_df[column_1].dropna()
            data_2 = model_df[column_2].dropna()

            statistic, p_value = mannwhitneyu(
                data_1,
                data_2,
                alternative="two-sided",
            )

            mean_1 = data_1.mean()
            mean_2 = data_2.mean()
            reported_p = max(p_value, 0.001)
            higher_mean = strategy_1 if mean_1 > mean_2 else strategy_2

            results.append({
                "Model": model,
                "Metric": metric_name,
                "Comparison": f"{strategy_1} vs {strategy_2}",
                "Mean 1": round(mean_1, 4),
                "Mean 2": round(mean_2, 4),
                "U-Statistic": round(statistic, 4),
                "P-Value": round(reported_p, 4),
                "Significant": "Yes" if reported_p < 0.05 else "No",
                "Higher Mean": higher_mean,
            })

mann_whitney_results = pd.DataFrame(results)
mann_whitney_results.to_excel(
    "Mann_Whitney_U_Results.xlsx",
    index=False,
)
display(mann_whitney_results)


## Model Heatmaps

In [ ]:

df = load_balanced_results(sample_size=2578, random_state=335)
metrics = ["BLEU", "ROUGE", "METEOR"]

for model in ["CodeLlama", "DeepSeek", "Qwen"]:
    model_df = df[df["Model"] == model]
    matrix = []

    for metric_index in [0, 1, 2]:
        row = [
            model_df[STRATEGIES[strategy][metric_index]].mean()
            for strategy in STRATEGIES
        ]
        matrix.append(row)

    matrix = np.array(matrix)

    fig, ax = plt.subplots(figsize=(10, 7), dpi=300)
    heatmap = ax.imshow(matrix, cmap="Pastel2")

    ax.set_xticks(np.arange(3))
    ax.set_yticks(np.arange(3))
    ax.set_xticklabels(
        ["Zero-shot", "Few-shot", "Chain of Thought"],
        fontsize=12,
        fontweight="bold",
    )
    ax.set_yticklabels(metrics, fontsize=12, fontweight="bold")

    for i in range(3):
        for j in range(3):
            ax.text(
                j,
                i,
                f"{matrix[i, j]:.4f}",
                ha="center",
                va="center",
                fontsize=12,
                fontweight="bold",
                color="black",
            )

    ax.set_title(
        f"Evaluation Score Comparison\n{model}",
        fontsize=18,
        fontweight="bold",
        pad=20,
    )

    ax.set_xticks(np.arange(-0.5, 3, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 3, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=3)

    for edge in ax.spines.values():
        edge.set_visible(False)

    colorbar = plt.colorbar(heatmap)
    colorbar.ax.tick_params(labelsize=10)

    plt.tight_layout()
    plt.savefig(
        f"Heatmap_{model}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close()


## Cross-Model Comparison

In [ ]:

df = load_balanced_results(sample_size=335, random_state=42)

comparison_results = []

for model in ["CodeLlama", "DeepSeek", "Qwen"]:
    model_df = df[df["Model"] == model]

    for strategy, columns in STRATEGIES.items():
        bleu = model_df[columns[0]].mean()
        rouge = model_df[columns[1]].mean()
        meteor = model_df[columns[2]].mean()
        overall = (bleu + rouge + meteor) / 3

        comparison_results.append({
            "Model": model,
            "Strategy": strategy,
            "BLEU": round(bleu, 4),
            "ROUGE": round(rouge, 4),
            "METEOR": round(meteor, 4),
            "Overall": round(overall, 4),
        })

comparison_df = pd.DataFrame(comparison_results)

labels = [
    f"{row['Model']}\n{row['Strategy']}"
    for _, row in comparison_df.iterrows()
]
values = comparison_df["Overall"].tolist()

colors = [
    "#A8DADC",
    "#BDE0FE",
    "#CDB4DB",
    "#FFD6A5",
    "#FFCAD4",
    "#D8E2DC",
    "#E9C46A",
    "#F4A261",
    "#B8F2E6",
]

fig, ax = plt.subplots(figsize=(14, 7), dpi=300)
bars = ax.bar(labels, values, color=colors)

for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.4f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color="black",
    )

ax.set_title(
    "Code Summary Quality Comparison Across Models",
    fontsize=18,
    fontweight="bold",
    pad=20,
)
ax.set_ylabel("Mean Evaluation Score", fontsize=12, fontweight="bold")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.xticks(rotation=10, fontsize=10, fontweight="bold")
plt.yticks(fontsize=10)
plt.tight_layout()
plt.savefig(
    "Cross_Model_Comparison.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.close()

heatmap_data = []

for model in ["CodeLlama", "DeepSeek", "Qwen"]:
    row = [
        comparison_df[
            (comparison_df["Model"] == model)
            & (comparison_df["Strategy"] == strategy)
        ]["Overall"].values[0]
        for strategy in STRATEGIES
    ]
    heatmap_data.append(row)

heatmap_data = np.array(heatmap_data)

fig, ax = plt.subplots(figsize=(10, 7), dpi=300)
heatmap = ax.imshow(heatmap_data, cmap="Pastel2")

ax.set_xticks(np.arange(3))
ax.set_yticks(np.arange(3))
ax.set_xticklabels(
    ["Zero-shot", "Few-shot", "Chain of Thought"],
    fontsize=12,
    fontweight="bold",
)
ax.set_yticklabels(
    ["CodeLlama", "DeepSeek", "Qwen"],
    fontsize=12,
    fontweight="bold",
)

for i in range(3):
    for j in range(3):
        ax.text(
            j,
            i,
            f"{heatmap_data[i, j]:.4f}",
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color="black",
        )

ax.set_title(
    "Cross-Model Comparison Heatmap",
    fontsize=18,
    fontweight="bold",
    pad=20,
)

ax.set_xticks(np.arange(-0.5, 3, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 3, 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=3)

for edge in ax.spines.values():
    edge.set_visible(False)

colorbar = plt.colorbar(heatmap)
colorbar.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.savefig(
    "Cross_Model_Comparison_Heatmap.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.close()
